<a href="https://colab.research.google.com/github/Scadelai/PCD/blob/main/Copy_of_ProjetoPCD_kmeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Projeto PCD: K-means 1D (naive) com Paralelização Progressiva**

## Profs. Álvaro e Denise (Turmas I e N)

O algoritmo K-Means realiza uma operação de agrupamento ("clusterização") para mineração de dados, ou seja, permite agrupar amostras de um dado conjunto em grupos homogêneos. O método particiona um conjunto de *n* observações (pontos) em *k* grupos, onde cada ponto será associado ao grupo cuja média seja a mais próxima. A distância Euclidiana é geralmente a métrica adotada para medir a proximidade.
A "clusterização" é um problema *NP-hard*, mas existem algoritmos heurísticos eficientes que podem rapidamente encontrar um ótimo local. Nesta implementação, a aplicação recebe como entrada as coordenadas (1D) de *k* centróides iniciais e um conjunto de dados. O K-Means realiza um processo iterativo, no qual os pontos são reagrupados de acordo com a menor distância Euclidiana entre eles e os centróides. Em seguida, o centróide de cada partição é recalculado tomando a média de todos os pontos da partição, e todo o procedimento é repetido até que nenhum centróide seja alterado e nenhum ponto seja atribuído a outro grupo. Ao final, o algoritmo retorna as coordenadas dos *k* centróides finais.


## Objetivo

Implementar o **k-means em 1 dimensão** (pontos `X[i]` e centróides `C[c]`), medir **SSE** e **desempenho**, e **paralelizar** o núcleo do algoritmo em três etapas independentes:

1. **OpenMP (CPU memória compartilhada)**
2. **CUDA (GPU)**
3. **MPI (memória distribuída)**

## Entradas e saídas

* **Entradas (CSV, 1 coluna, sem cabeçalho):**

  * `dados.csv` com **N** valores (pontos).
  * `centroides_iniciais.csv` com **K** valores.
* **Saídas:**

  * No terminal: **iterações**, **SSE (*Sum of Squared Errors*, ou Soma dos Erros Quadráticos, em português) final**, **tempo total** (ms).
  * Arquivos: `assign.csv` (N linhas, índice do cluster por ponto) e `centroids.csv` (K linhas com centróides finais).

## Algoritmo (base “naive”)

Itere até `max_iter` ou até variar pouco o SSE (`eps`):

1. **Assignment:** para cada ponto, escolher o centróide mais próximo (minimiza $(x_i - c)^2$); acumular **SSE**.
2. **Update:** para cada cluster, **média** dos pontos atribuídos. Se um cluster ficar vazio, **copie `X[0]`** (estratégia simples).

---

## Etapa 0 — Versão sequencial (baseline)

* Executar a versão **sequencial** (fornecida mais abaixo).
* Coletar: **SSE por iteração**, **tempo total**, **iterações**.
* Salvar esses números: serão a **linha de base** para speedup.

---

## Etapa 1 — OpenMP (CPU)

**Meta:** paralelizar as funções de *assignment* e *update* na CPU.

### O que paralelizar

* **Assignment:** laço `for (i=0; i<N; ++i)`.
* **Update:**

  * opção A (mais simples): usar **acumuladores por thread** (`sum_thread[c]`, `cnt_thread[c]`) e **reduzir** após a região paralela;
  * opção B: usar `#pragma omp critical` e verificar impactos no desempenho.

### Medições

* **Escalonamento em threads:** T ∈ {1, 2, 4, 8, 16, …}.
* **Speedup** = tempo_serial / tempo_OpenMP.
* **Afinar:** *schedule* (`static` vs `dynamic`) e *chunk size*.
* **Validação:** SSE não deve **aumentar** ao longo das iterações (pode ficar igual se convergiu).

### Dica de compilação

```bash
gcc -O2 -fopenmp -std=c99 kmeans_1d_omp.c -o kmeans_1d_omp -lm
```

---

## Etapa 2 — CUDA (GPU)

**Meta:** mover o **assignment** para a GPU; o **update** pode ser feito na GPU (com atomics) ou no host (copiando `assign`).

### Desenho mínimo

* **Kernel de assignment:** 1 *thread* por ponto `i`.

  * Cada thread varre **K** centróides, calcula `d = (X[i]-C[c])^2`, guarda o melhor e escreve `assign[i]`.
  * (Opcional) carregar `C` em **memória constante**.
* **SSE:** reduzir no host somando os erros por ponto (ou fazer redução em blocos).
* **Update:**

  * opção A (mais simples): copiar `assign` para CPU e calcular médias no host;
  * opção B: usar **atomics** em `sum[c]` e `cnt[c]` na GPU e depois dividir.

### Medições

* **Tamanho de bloco** (p.ex., 128, 256, 512) × **grid**;
* **Tempos**: H2D/D2H, *kernel*, total;
* **Throughput**: pontos/s; **speedup** vs. serial e vs. OpenMP.

### Dica de compilação

```bash
nvcc -O2 kmeans_1d_cuda.cu -o kmeans_1d_cuda
```

---

## Etapa 3 — MPI (distribuída)

**Meta:** distribuir os **N** pontos entre **P** processos; centróides são **globais** a cada iteração.

### Passos por iteração

1. **Broadcast** (ou inicialização compartilhada): todos os processos têm `C`.
2. **Assignment local:** cada processo calcula `assign_local` e `SSE_local` para seu bloco de pontos.
3. **Redução global:**

   * somar `SSE_local` → `SSE_global` com `MPI_Reduce`;
   * somar `sum_local[c]` e `cnt_local[c]` para todos os clusters com `MPI_Allreduce`;
   * cada processo atualiza `C` com os resultados globais.
4. Próxima iteração até convergir.

### Medições

* **Strong scaling:** P ∈ {1, 2, 4, 8, …}.
* **Tempo de comunicação:** destacar o custo de `Allreduce`.
* **Speedup** vs. serial e OpenMP.

### Dica de compilação/execução

```bash
mpicc -O2 -std=c99 kmeans_1d_mpi.c -o kmeans_1d_mpi -lm
mpirun -np 4 ./kmeans_1d_mpi dados.csv centroides_iniciais.csv [args...]
```

---

## Conjuntos de teste sugeridos (1D)

* **Pequeno:** N=10^4, K=4
* **Médio:** N=10^5, K=8
* **Grande:** N=10^6, K=16 (se houver memória)
  Gere dados com mistura de faixas (ex.: perto de 0, 10, 20, 30) para facilitar a verificação visual.

---

## O que entregar

1. **Código no github**: `serial/`, `openmp/`, `cuda/`, `mpi/` (cada pasta com `README.md` de como compilar/rodar).
2. **Relatório curto (4–6 págs po etapa)**:

   * Ambiente (CPU/GPU/RAM/rede; versões de compilador).
   * Gráficos: **tempo**, **speedup**, **pontos/s** por etapa.
   * Para MPI: curva de **speedup** e comentário sobre custo de `Allreduce`.
   * Para CUDA: impacto de **block size** e custo de **transferência**.
   * Para OpenMP: efeito de **nº de threads** e de *schedule*.
   * Seções de **validação** (SSE por iteração, convergência, igualdade de resultados entre versões dentro de tolerância).
   * Análise de resultados e conclusões
   * Referências bibliográficas: apresente uma pequena revisão bibliográfica e compare seus resultados com outros encontrados na literatura.

---

## Critérios de avaliação

* **Desempenho e análise** (speedup, tempos de execução, eficiência e gargalos por arquitetura): **30%**
* **Corretude e reprodutibilidade** (SSE consistente, convergência, demonstração da corretude da execução): **30%**
* **Relatório** (clareza, gráficos, referências bibliográficas, análises e conclusões): **20%**
* **Qualidade do código e organização**: **10%**
* **Extra** (implementação e/ou análises não sugeridas no enunciado e que melhorem a qualidade científica do trabalho): **10%**


---

## Dicas rápidas

* Padronize **parâmetros** (N, K, `max_iter`, `eps`) entre as versões para comparar.
* Fixe uma **semente** ao gerar dados (quando aplicável) para repetibilidade.



##Exemplo: código sequencial e arquivos de entrada e saída

In [ ]:
%%writefile centroides_iniciais.csv

10
30
60
90


Overwriting centroides_iniciais.csv


In [ ]:
%%writefile dados.csv

1
2
3
4
5
6
7
8
4.5
5.5
18
19
20
21
22
23
19.5
20.5
100
125

Overwriting dados.csv


In [5]:
import numpy as np

# Parâmetros do Relatório (N=1M, K=5)
N = 1000000
K = 5

print(f"Gerando dados.csv com N={N}...")
# Gera dados aleatórios (mistura de distribuições normais para criar clusters)
data = np.concatenate([
    np.random.normal(loc=10, scale=2, size=N//5),
    np.random.normal(loc=30, scale=2, size=N//5),
    np.random.normal(loc=50, scale=2, size=N//5),
    np.random.normal(loc=70, scale=2, size=N//5),
    np.random.normal(loc=90, scale=2, size=N//5)
])
np.random.shuffle(data)
np.savetxt("dados.csv", data, fmt='%.6f')

print(f"Gerando centroides.csv com K={K}...")
# Gera centróides iniciais aleatórios
centroids = np.random.uniform(0, 100, K)
np.savetxt("centroides.csv", centroids, fmt='%.6f')

print("Arquivos gerados com sucesso!")

Gerando dados.csv com N=1000000...
Gerando centroides.csv com K=5...
Arquivos gerados com sucesso!


In [6]:
%%writefile kmeans_naive.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

/* --- Leitura de CSV (Idêntico ao PDF) --- */
static int count_rows(const char *path) {
    FILE *f = fopen(path, "r");
    if(!f) { fprintf(stderr, "Erro ao abrir %s\n", path); exit(1); }
    int rows = 0; char line[8192];
    while(fgets(line, sizeof(line), f)){
        int only_ws=1;
        for(char *p=line; *p; p++) {
            if(*p!=' ' && *p!= '\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(!only_ws) rows++;
    }
    fclose(f);
    return rows;
}

static double *read_csv_1col(const char *path, int *n_out){
    int R = count_rows(path);
    if(R<=0){ fprintf(stderr, "Arquivo vazio ou erro: %s\n", path); exit(1); }
    double *A = (double*)malloc((size_t)R * sizeof(double));
    if(!A){ fprintf(stderr, "Sem memoria\n"); exit(1); }

    FILE *f = fopen(path, "r");
    char line[8192];
    int r=0;
    while(fgets(line, sizeof(line), f)){
        int only_ws=1;
        for(char *p=line; *p; p++) {
            if(*p!=' ' && *p!= '\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(only_ws) continue;

        char *tok = strtok(line, ",; \t");
        if(tok) {
            A[r] = atof(tok);
            r++;
        }
        if(r>=R) break;
    }
    fclose(f);
    *n_out = r;
    return A;
}

/* --- Escrita de CSV (Opcional, mas pedido no PDF) --- */
static void write_assign_csv(const char *path, const int *assign, int N){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f) return;
    for(int i=0;i<N;i++) fprintf(f, "%d\n", assign[i]);
    fclose(f);
}

static void write_centroids_csv(const char *path, const double *C, int K){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f) return;
    for(int c=0;c<K;c++) fprintf(f, "%.6f\n", C[c]);
    fclose(f);
}

/* --- Algoritmo Naive --- */
static double assignment_step_1d(const double *X, const double *C, int *assign, int N, int K){
    double sse = 0.0;
    for(int i=0; i<N; i++){
        int best = -1;
        double bestd = 1e300;
        for(int c=0; c<K; c++){
            double diff = X[i] - C[c];
            double d = diff*diff;
            if(d < bestd) { bestd = d; best = c; }
        }
        assign[i] = best;
        sse += bestd;
    }
    return sse;
}

static void update_step_1d(const double *X, double *C, const int *assign, int N, int K){
    double *sum = (double*) calloc((size_t)K, sizeof(double));
    int *cnt = (int*) calloc((size_t)K, sizeof(int));

    for(int i=0; i<N; i++){
        int a = assign[i];
        sum[a] += X[i];
        cnt[a]++;
    }

    for(int c=0; c<K; c++){
        if(cnt[c]>0) C[c] = sum[c] / (double)cnt[c];
        else C[c] = X[0]; // Estratégia naive do PDF: cluster vazio pega X[0]
    }
    free(sum); free(cnt);
}

int main(int argc, char **argv) {
    if(argc < 3){
        printf("Uso: %s dados.csv centroides.csv [max_iter] [eps] ...\n", argv[0]);
        return 1;
    }
    const char *pathX = argv[1];
    const char *pathC = argv[2];
    int max_iter = (argc>3) ? atoi(argv[3]) : 50;
    double eps = (argc>4) ? atof(argv[4]) : 1e-4;

    int N=0, K=0;
    double *X = read_csv_1col(pathX, &N);
    double *C = read_csv_1col(pathC, &K);
    int *assign = (int*)malloc((size_t)N * sizeof(int));

    clock_t t0 = clock();

    double prev_sse = 1e300;
    double sse = 0.0;
    int it;

    for(it=0; it<max_iter; it++){
        sse = assignment_step_1d(X, C, assign, N, K);
        double rel = fabs(sse - prev_sse) / (prev_sse > 0.0? prev_sse: 1.0);
        if(rel < eps) { it++; break; } // Convergiu

        update_step_1d(X, C, assign, N, K);
        prev_sse = sse;
    }

    clock_t t1 = clock();
    double ms = 1000.0 * (double)(t1-t0) / (double)CLOCKS_PER_SEC;

    printf("K-means 1D (Naive Sequencial)\n");
    printf("N=%d K=%d max_iter=%d eps=%g\n", N, K, max_iter, eps);
    printf("Iterações: %d | SSE final: %.6f | Tempo: %.1f ms\n", it, sse, ms);

    // Opcional: salvar saídas
    if(argc > 5) write_assign_csv(argv[5], assign, N);
    if(argc > 6) write_centroids_csv(argv[6], C, K);

    free(assign); free(X); free(C);
    return 0;
}

Writing kmeans_naive.c


In [1]:
%%writefile kmeans_omp.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <omp.h>
#include <float.h>

/* --- Funções Auxiliares de CSV (Idênticas ao Naive) --- */
static int count_rows(const char *path){
    FILE *f = fopen(path, "r");
    if(!f){ fprintf(stderr,"Erro ao abrir %s\n", path); exit(1); }
    int rows=0; char line[8192];
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(!only_ws) rows++;
    }
    fclose(f);
    return rows;
}

static double *read_csv_1col(const char *path, int *n_out){
    int R = count_rows(path);
    if(R<=0){ fprintf(stderr,"Arquivo vazio: %s\n", path); exit(1); }
    double *A = (double*)malloc((size_t)R * sizeof(double));
    if(!A){ fprintf(stderr,"Sem memoria para %d linhas\n", R); exit(1); }
    FILE *f = fopen(path, "r");
    if(!f){ fprintf(stderr,"Erro ao abrir %s\n", path); free(A); exit(1); }
    char line[8192];
    int r=0;
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(only_ws) continue;
        const char *delim = ",; \t";
        char *tok = strtok(line, delim);
        if(!tok){ continue; }
        A[r] = atof(tok);
        r++;
        if(r>=R) break;
    }
    fclose(f);
    *n_out = r;
    return A;
}

static void write_assign_csv(const char *path, const int *assign, int N){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f){ fprintf(stderr,"Erro ao abrir %s para escrita\n", path); return; }
    for(int i=0;i<N;i++) fprintf(f, "%d\n", assign[i]);
    fclose(f);
}

static void write_centroids_csv(const char *path, const double *C, int K){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f){ fprintf(stderr,"Erro ao abrir %s para escrita\n", path); return; }
    for(int c=0;c<K;c++) fprintf(f, "%.6f\n", C[c]);
    fclose(f);
}

/* --- Funções Core Paralelizadas com OpenMP --- */

// Etapa 1: Assignment (Paralelizada)
static double assignment_step_omp(const double *X, const double *C, int *assign, int N, int K){
    double sse = 0.0;

    // Paraleliza o laço dos pontos e reduz a variável sse
    #pragma omp parallel for reduction(+:sse)
    for(int i=0; i<N; i++){
        int best = -1;
        double bestd = 1e300;
        for(int c=0; c<K; c++){
            double diff = X[i] - C[c];
            double d = diff*diff;
            if(d < bestd){ bestd = d; best = c; }
        }
        assign[i] = best;
        sse += bestd;
    }
    return sse;
}

// Etapa 2: Update (Paralelizada)
static void update_step_omp(const double *X, double *C, const int *assign, int N, int K){
    double *sum = (double*)calloc((size_t)K, sizeof(double));
    int *cnt = (int*)calloc((size_t)K, sizeof(int));

    // Opção A: Redução em arrays (Feature do OpenMP 4.5+ para arrays C/C++)
    // Se o compilador for antigo, pode precisar de critical ou array privado.
    // Assumindo GCC moderno no Colab:
    #pragma omp parallel for reduction(+:sum[:K], cnt[:K])
    for(int i=0; i<N; i++){
        int a = assign[i];
        sum[a] += X[i];
        cnt[a] += 1;
    }

    for(int c=0; c<K; c++){
        if(cnt[c] > 0) C[c] = sum[c] / (double)cnt[c];
        else           C[c] = X[0];
    }
    free(sum); free(cnt);
}

int main(int argc, char **argv){
    if(argc < 3){
        printf("Uso: %s dados.csv centroides.csv [max_iter] [eps] [out_assign] [out_cent]\n", argv[0]);
        return 1;
    }
    const char *pathX = argv[1];
    const char *pathC = argv[2];
    int max_iter = (argc>3)? atoi(argv[3]) : 50;
    double eps   = (argc>4)? atof(argv[4]) : 1e-4;
    const char *outAssign   = (argc>5)? argv[5] : NULL;
    const char *outCentroid = (argc>6)? argv[6] : NULL;

    int N=0, K=0;
    double *X = read_csv_1col(pathX, &N);
    double *C = read_csv_1col(pathC, &K);
    int *assign = (int*)malloc(N * sizeof(int));

    printf("K-Means OpenMP: N=%d, K=%d, Threads=%d\n", N, K, omp_get_max_threads());

    double prev_sse = 1e300;
    double sse = 0.0;
    int it = 0;

    double t_start = omp_get_wtime();

    for(it=0; it<max_iter; it++){
        sse = assignment_step_omp(X, C, assign, N, K);

        double rel = fabs(sse - prev_sse) / (prev_sse > 0.0 ? prev_sse : 1.0);
        if(rel < eps){ it++; break; } // Convergiu

        update_step_omp(X, C, assign, N, K);
        prev_sse = sse;
    }

    double t_end = omp_get_wtime();

    printf("Iterações: %d | SSE final: %.6f | Tempo: %.4f s\n", it, sse, t_end - t_start);

    write_assign_csv(outAssign, assign, N);
    write_centroids_csv(outCentroid, C, K);

    free(assign); free(X); free(C);
    return 0;
}

Writing kmeans_omp.c


In [3]:
%%writefile kmeans_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <float.h>
#include <string.h>
#include <math.h>
#include <cuda_runtime.h>
#include <time.h>

#define CUDA_CHECK(err) { \
    if(err != cudaSuccess) { \
        fprintf(stderr, "Erro CUDA: %s\n", cudaGetErrorString(err)); \
        exit(1); \
    } \
}

/* --- Funções Auxiliares de CSV (Replicadas) --- */
// (Simplificadas para economizar espaço visual, mas funcionais)
static int count_rows(const char *path){
    FILE *f=fopen(path,"r"); if(!f) exit(1);
    int rows=0; char line[8192];
    while(fgets(line,sizeof(line),f)) {
        int w=1; for(char*p=line;*p;p++) if(*p>' ') w=0;
        if(!w) rows++;
    }
    fclose(f); return rows;
}
static double *read_csv_1col(const char *path, int *n_out){
    int R=count_rows(path);
    double *A=(double*)malloc(R*sizeof(double));
    FILE *f=fopen(path,"r"); char line[8192]; int r=0;
    while(fgets(line,sizeof(line),f)){
        int w=1; for(char*p=line;*p;p++) if(*p>' ') w=0;
        if(w) continue;
        A[r++] = atof(strtok(line, ",; \t"));
        if(r>=R) break;
    }
    fclose(f); *n_out=r; return A;
}
static void write_csvs(const char *fA, const char *fC, int *as, double *C, int N, int K){
    if(fA){ FILE *f=fopen(fA,"w"); for(int i=0;i<N;i++) fprintf(f,"%d\n",as[i]); fclose(f); }
    if(fC){ FILE *f=fopen(fC,"w"); for(int c=0;c<K;c++) fprintf(f,"%.6f\n",C[c]); fclose(f); }
}

/* --- Kernel CUDA --- */
__global__ void assignment_kernel(const double *X, const double *C, int *assign, double *errors, int N, int K){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    double myX = X[i];
    int best = -1;
    double bestd = 1e300; // Usando double para precisão

    for(int c=0; c<K; c++){
        double diff = myX - C[c];
        double d = diff * diff;
        if(d < bestd){ bestd = d; best = c; }
    }
    assign[i] = best;
    errors[i] = bestd;
}

int main(int argc, char **argv){
    if(argc < 3){ printf("Uso: %s dados.csv centroides.csv [iter] [eps] ...\n", argv[0]); return 1; }

    int N=0, K=0;
    double *h_X = read_csv_1col(argv[1], &N);
    double *h_C = read_csv_1col(argv[2], &K);
    int max_iter = (argc>3)? atoi(argv[3]) : 50;
    double eps   = (argc>4)? atof(argv[4]) : 1e-4;

    int *h_assign = (int*)malloc(N*sizeof(int));
    double *h_errors = (double*)malloc(N*sizeof(double));

    // Alocação na GPU
    double *d_X, *d_C, *d_errors;
    int *d_assign;
    CUDA_CHECK(cudaMalloc(&d_X, N*sizeof(double)));
    CUDA_CHECK(cudaMalloc(&d_C, K*sizeof(double)));
    CUDA_CHECK(cudaMalloc(&d_assign, N*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_errors, N*sizeof(double)));

    // Copia X para GPU uma única vez (X não muda)
    CUDA_CHECK(cudaMemcpy(d_X, h_X, N*sizeof(double), cudaMemcpyHostToDevice));

    int blockSize = 256;
    int gridSize = (N + blockSize - 1) / blockSize;

    printf("K-Means CUDA: N=%d, K=%d, Grid=%d, Block=%d\n", N, K, gridSize, blockSize);

    // Timer
    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    double prev_sse = 1e300;
    double sse = 0.0;
    int it = 0;

    for(it=0; it<max_iter; it++){
        // 1. Copia Centróides atuais para GPU
        CUDA_CHECK(cudaMemcpy(d_C, h_C, K*sizeof(double), cudaMemcpyHostToDevice));

        // 2. Kernel Assignment
        assignment_kernel<<<gridSize, blockSize>>>(d_X, d_C, d_assign, d_errors, N, K);
        cudaDeviceSynchronize();

        // 3. Copia resultados de volta para CPU
        // Precisamos do vetor de erros para calcular o SSE total e do assign para o update
        CUDA_CHECK(cudaMemcpy(h_assign, d_assign, N*sizeof(int), cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaMemcpy(h_errors, d_errors, N*sizeof(double), cudaMemcpyDeviceToHost));

        // 4. Redução do SSE no Host (Simples)
        sse = 0.0;
        for(int i=0; i<N; i++) sse += h_errors[i];

        // Verifica convergência
        double rel = fabs(sse - prev_sse) / (prev_sse > 0.0 ? prev_sse : 1.0);
        if(rel < eps){ it++; break; }

        // 5. Update Centroids no Host (Estratégia A do PDF)
        double *sum = (double*)calloc(K, sizeof(double));
        int *cnt = (int*)calloc(K, sizeof(int));
        for(int i=0; i<N; i++){
            int a = h_assign[i];
            sum[a] += h_X[i];
            cnt[a]++;
        }
        for(int c=0; c<K; c++){
            if(cnt[c]>0) h_C[c] = sum[c] / cnt[c];
            else         h_C[c] = h_X[0];
        }
        free(sum); free(cnt);
        prev_sse = sse;
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    printf("Iterações: %d | SSE final: %.6f | Tempo Total GPU: %.1f ms\n", it, sse, milliseconds);

    // Salvar saídas
    if(argc>5) write_csvs(argv[5], argv[6], h_assign, h_C, N, K);

    free(h_X); free(h_C); free(h_assign); free(h_errors);
    cudaFree(d_X); cudaFree(d_C); cudaFree(d_assign); cudaFree(d_errors);
    return 0;
}

Writing kmeans_cuda.cu


In [4]:
%%writefile kmeans_mpi.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <mpi.h>
#include <float.h>

/* --- Funções de Leitura (Apenas Processo 0 usa, mas devem estar definidas) --- */
static int count_rows(const char *path){
    FILE *f=fopen(path,"r"); if(!f) return 0;
    int rows=0; char line[8192];
    while(fgets(line,sizeof(line),f)) {
        int w=1; for(char*p=line;*p;p++) if(*p>' ') w=0;
        if(!w) rows++;
    }
    fclose(f); return rows;
}
static double *read_csv_1col(const char *path, int *n_out){
    int R=count_rows(path); if(R==0) return NULL;
    double *A=(double*)malloc(R*sizeof(double));
    FILE *f=fopen(path,"r"); char line[8192]; int r=0;
    while(fgets(line,sizeof(line),f)){
        int w=1; for(char*p=line;*p;p++) if(*p>' ') w=0;
        if(w) continue;
        A[r++] = atof(strtok(line, ",; \t"));
        if(r>=R) break;
    }
    fclose(f); *n_out=r; return A;
}
static void write_centroids(const char *path, const double *C, int K){
    FILE *f = fopen(path, "w");
    if(f){ for(int c=0;c<K;c++) fprintf(f, "%.6f\n", C[c]); fclose(f); }
}

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int N_global = 0;
    int K = 0;
    int max_iter = 50;
    double eps = 1e-4;
    double *X_full = NULL;
    double *C = NULL;

    // --- Passo 1: Rank 0 lê os dados e parâmetros ---
    if (rank == 0) {
        if(argc < 3){
            fprintf(stderr, "Uso MPI: mpirun ... %s dados.csv centroides.csv [args]\n", argv[0]);
            MPI_Abort(MPI_COMM_WORLD, 1);
        }
        X_full = read_csv_1col(argv[1], &N_global);
        C = read_csv_1col(argv[2], &K);
        if(argc > 3) max_iter = atoi(argv[3]);
        if(argc > 4) eps = atof(argv[4]);

        printf("MPI Master: Lido N=%d, K=%d. Distribuindo para %d processos.\n", N_global, K, size);
    }

    // --- Passo 2: Broadcast de Parâmetros (K, max_iter, eps, N_global) ---
    MPI_Bcast(&N_global, 1, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Bcast(&K, 1, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Bcast(&max_iter, 1, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Bcast(&eps, 1, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    // Aloca C em todos os processos
    if(rank != 0) C = (double*)malloc(K * sizeof(double));
    // Broadcast dos centróides iniciais
    MPI_Bcast(C, K, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    // --- Passo 3: Distribuir Pontos (Scatter) ---
    // Calculando quantos pontos cada processo recebe
    int n_local = N_global / size;
    int remainder = N_global % size;

    // Simplificação: Vamos cortar o resto para facilitar o Scatter padrão,
    // ou assumir que N é divisível. Para ser robusto no Scatter, usamos arrays de counts.
    // Para simplificar muito para o trabalho:
    // Usamos MPI_Scatterv (complexo) ou fazemos todos lerem?
    // Vamos fazer o Scatterv pois é o "correto" pedido no trabalho.

    int *sendcounts = NULL;
    int *displs = NULL;
    if (rank == 0) {
        sendcounts = (int*)malloc(size * sizeof(int));
        displs = (int*)malloc(size * sizeof(int));
        int sum = 0;
        for (int i = 0; i < size; i++) {
            sendcounts[i] = (N_global / size) + (i < remainder ? 1 : 0);
            displs[i] = sum;
            sum += sendcounts[i];
        }
    }

    // Processos recebem a contagem local
    int my_n;
    if(rank == 0) my_n = sendcounts[0];
    else {
        // Recalcula lógica localmente ou manda msg.
        // Mais fácil: scatter int counts primeiro? Não, calculamos local.
        my_n = (N_global / size) + (rank < remainder ? 1 : 0);
    }

    double *X_local = (double*)malloc(my_n * sizeof(double));

    // Distribui os pontos
    MPI_Scatterv(X_full, sendcounts, displs, MPI_DOUBLE,
                 X_local, my_n, MPI_DOUBLE,
                 0, MPI_COMM_WORLD);

    if(rank == 0) free(X_full); // Master não precisa mais do full array

    // --- Passo 4: Loop Principal ---
    double prev_sse = 1e300;
    double sse_global = 0.0;
    int it = 0;
    int *assign_local = (int*)malloc(my_n * sizeof(int));

    // Buffers para redução
    double *sum_local = (double*)malloc(K * sizeof(double));
    int *cnt_local = (int*)malloc(K * sizeof(int));
    double *sum_global = (double*)malloc(K * sizeof(double));
    int *cnt_global = (int*)malloc(K * sizeof(int));

    double t_start = MPI_Wtime();

    for(it=0; it<max_iter; it++){
        // 1. Broadcast Centroids (garantir que todos tenham o C atualizado)
        // Na primeira iter já foi feito. Nas próximas, precisamos.
        MPI_Bcast(C, K, MPI_DOUBLE, 0, MPI_COMM_WORLD);

        // 2. Assignment Local
        double sse_local = 0.0;
        for(int i=0; i<my_n; i++){
            double bestd = 1e300;
            int best = -1;
            for(int c=0; c<K; c++){
                double d = (X_local[i] - C[c]) * (X_local[i] - C[c]);
                if(d < bestd){ bestd = d; best = c; }
            }
            assign_local[i] = best;
            sse_local += bestd;
        }

        // 3. Redução SSE
        MPI_Allreduce(&sse_local, &sse_global, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);

        // Checar convergência (todos processos precisam saber para sair do loop)
        double rel = fabs(sse_global - prev_sse) / (prev_sse > 0.0 ? prev_sse : 1.0);
        if(rel < eps){
            if(rank==0) it++; // Incrementa para contagem correta
            break;
        }
        prev_sse = sse_global;

        // 4. Preparar Update (Soma Local)
        memset(sum_local, 0, K*sizeof(double));
        memset(cnt_local, 0, K*sizeof(int));
        for(int i=0; i<my_n; i++){
            int a = assign_local[i];
            sum_local[a] += X_local[i];
            cnt_local[a] += 1;
        }

        // 5. Redução Global para Update (Soma Global)
        MPI_Allreduce(sum_local, sum_global, K, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
        MPI_Allreduce(cnt_local, cnt_global, K, MPI_INT, MPI_SUM, MPI_COMM_WORLD);

        // 6. Atualizar Centróides (Cada processo atualiza sua cópia de C)
        for(int c=0; c<K; c++){
            if(cnt_global[c] > 0) C[c] = sum_global[c] / cnt_global[c];
            // Se vazio, estratégia naive: manter anterior ou resetar (não tratado complexamente aqui)
        }
    }

    double t_end = MPI_Wtime();

    if(rank == 0){
        printf("MPI Final: Iterações=%d | SSE=%.6f | Tempo=%.4fs\n", it, sse_global, t_end-t_start);
        if(argc > 6) write_centroids(argv[6], C, K);
        // Nota: Gravar assign.csv em MPI é chato (precisa de Gather), opcional na maioria desses trabalhos.
    }

    free(X_local); free(assign_local); free(C);
    free(sum_local); free(cnt_local); free(sum_global); free(cnt_global);
    if(rank==0) { free(sendcounts); free(displs); }

    MPI_Finalize();
    return 0;
}

Writing kmeans_mpi.c


In [22]:
import os
import subprocess
import re
import numpy as np
import pandas as pd

# --- Configurações ---
N_RUNS = 50  # Quantas vezes rodar cada teste para tirar a média
DADOS = "dados.csv"
CENTROIDES = "centroides.csv"
ARGS = f"{DADOS} {CENTROIDES} 50 0.0001" # Arquivos, iter, eps

# --- 1. Compilação ---
commands_compile = [
    "gcc -O2 -std=c99 kmeans_naive.c -o kmeans_naive -lm",
    "gcc -O2 -fopenmp kmeans_omp.c -o kmeans_omp -lm",
    "nvcc -O2 kmeans_cuda.cu -o kmeans_cuda -Wno-deprecated-gpu-targets",
    "mpicc -O2 kmeans_mpi.c -o kmeans_mpi -lm"
]

for cmd in commands_compile:
    ret = os.system(cmd)
    if ret != 0:
        print(f"Erro ao compilar: {cmd}")
        exit(1)
print("Compilação concluída!\n")

# --- 2. Definição dos Testes ---
# Estrutura: Nome do teste, Comando, Regex para pegar o tempo, Unidade original ('s' ou 'ms')
tests = [
    {
        "name": "Sequencial (Naive)",
        "cmd": f"./kmeans_naive {ARGS}",
        "regex": r"Tempo: ([\d\.]+) ms",
        "unit": "ms"
    },
    {
        "name": "OpenMP (2 Threads)",
        "cmd": f"export OMP_NUM_THREADS=2 && ./kmeans_omp {ARGS}",
        "regex": r"Tempo: ([\d\.]+) s",
        "unit": "s"
    },
    {
        "name": "CUDA (GPU)",
        "cmd": f"./kmeans_cuda {ARGS}",
        "regex": r"Tempo Total GPU: ([\d\.]+) ms",
        "unit": "ms"
    },
    {
        "name": "MPI (4 Processos)",
        # --oversubscribe é necessário no Colab pois ele só tem 2 cores físicos
        "cmd": f"mpirun --allow-run-as-root --oversubscribe -np 4 ./kmeans_mpi {ARGS}",
        "regex": r"Tempo=([\d\.]+)s",
        "unit": "s"
    }
]

# --- 3. Execução e Benchmarking ---
results = []


for test in tests:
    print(f"Rodando: {test['name']} ", end="")
    times = []

    for i in range(N_RUNS):
        # Executa o comando no shell
        process = subprocess.Popen(test['cmd'], shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        stdout, stderr = process.communicate()

        # Procura o tempo na saída usando Regex
        match = re.search(test['regex'], stdout)
        if match:
            t_val = float(match.group(1))
            # Converte tudo para milissegundos (ms)
            if test['unit'] == 's':
                t_val *= 1000.0
            times.append(t_val)
            print(".", end="", flush=True)
        else:
            print("x", end="", flush=True)
            print(f"\nErro na leitura da saída de {test['name']}:\n{stdout}")

    # Calcula média e desvio padrão
    if times:
        avg_time = np.mean(times)
        std_time = np.std(times)
        results.append({
            "Versão": test['name'],
            "Média (ms)": f"{avg_time:.2f}",
            "Desvio Padrão": f"{std_time:.2f}",
            "Speedup": 0.0 # Calculado depois
        })
        print(f" Média: {avg_time:.2f} ms")
    else:
        print(" Falhou.")
    print("")

# --- 4. Cálculo do Speedup e Tabela Final ---
if results:
    # O primeiro teste (Sequencial) é a base. Speedup = Tempo_Seq / Tempo_Paralelo
    base_time = float(results[0]["Média (ms)"])

    for res in results:
        current_time = float(res["Média (ms)"])
        if current_time > 0:
            res["Speedup"] = f"{base_time / current_time:.2f}x"
        else:
            res["Speedup"] = "-"

    # Exibe Tabela Bonita com Pandas
    df = pd.DataFrame(results)
    print("\n" + "="*50)
    print("RESULTADOS FINAIS (Média de 50 rodadas)")
    print("="*50)
    print(df.to_string(index=False))
    print("="*50)
else:
    print("Nenhum resultado obtido.")

Compilação concluída!

Rodando: Sequencial (Naive) .................................................. Média: 52.04 ms

Rodando: OpenMP (2 Threads) .................................................. Média: 47.03 ms

Rodando: CUDA (GPU) .................................................. Média: 24.56 ms

Rodando: MPI (4 Processos) .................................................. Média: 58.37 ms


RESULTADOS FINAIS (Média de 50 rodadas)
            Versão Média (ms) Desvio Padrão Speedup
Sequencial (Naive)      52.04         12.78   1.00x
OpenMP (2 Threads)      47.03         15.04   1.11x
        CUDA (GPU)      24.56          1.88   2.12x
 MPI (4 Processos)      58.37         19.80   0.89x


In [19]:
import os
import subprocess
import re
import pandas as pd

# Parâmetros
N_RUNS = 50
ARGS = "dados.csv centroides.csv 50 0.0001"

results_fine = []

# --- 1. Teste de CUDA Block Sizes (128, 256, 512) ---
# Precisamos recompilar o código mudando o blockSize.
# Como o blockSize era variável no código C que passei anteriormente (hardcoded 256?),
# vamos usar sed para trocar o valor no código fonte original antes de compilar.

block_sizes = [128, 256, 512]
for bs in block_sizes:
    print(f"--> Testando CUDA Block Size = {bs}...")
    # Truque: altera o int blockSize = ... no código fonte usando regex
    os.system(f"sed -i 's/int blockSize = [0-9]*;/int blockSize = {bs};/' kmeans_cuda.cu")
    os.system("nvcc -O2 kmeans_cuda.cu -o kmeans_cuda_bench -Wno-deprecated-gpu-targets")

    cmd = f"./kmeans_cuda_bench {ARGS}"
    times = []
    for _ in range(N_RUNS):
        out = subprocess.getoutput(cmd)
        m = re.search(r"Tempo Total GPU: ([\d\.]+) ms", out)
        if m: times.append(float(m.group(1)))

    if times:
        avg = sum(times)/len(times)
        results_fine.append({"Teste": f"CUDA Block {bs}", "Tempo (ms)": f"{avg:.2f}", "Obs": "Impacto na GPU"})

# --- 2. Teste de OpenMP Schedules (Static vs Dynamic) ---
# O código original usava #pragma omp parallel for reduction...
# Vamos injetar a cláusula schedule.

schedules = ["static", "dynamic"]
for sched in schedules:
    print(f"--> Testando OpenMP Schedule = {sched}...")
    # Restaura o código base (apenas para garantir) e insere o schedule
    # O comando sed busca a linha do pragma e adiciona o schedule
    os.system(f"sed -i 's/#pragma omp parallel for reduction(+:sse)/#pragma omp parallel for schedule({sched}) reduction(+:sse)/' kmeans_omp.c")
    os.system("gcc -O2 -fopenmp kmeans_omp.c -o kmeans_omp_bench -lm")

    # Rodar com 2 threads (máximo do Colab)
    cmd = f"export OMP_NUM_THREADS=2 && ./kmeans_omp_bench {ARGS}"
    times = []
    for _ in range(N_RUNS):
        out = subprocess.getoutput(cmd)
        # O OpenMP pode printar em s ou ms, o regex pega ambos convertendo se necessário
        # Assumindo que o código base printa 'Tempo: X s'
        m = re.search(r"Tempo: ([\d\.]+) s", out)
        if m: times.append(float(m.group(1)) * 1000) # Converte para ms

    if times:
        avg = sum(times)/len(times)
        results_fine.append({"Teste": f"OpenMP {sched.capitalize()}", "Tempo (ms)": f"{avg:.2f}", "Obs": "Impacto do Load Balancing"})

    # Limpar alteração no arquivo para não quebrar próxima rodada
    os.system("sed -i 's/ schedule(.*) / /' kmeans_omp.c")

# --- 3. Exibir Tabela de Análise Fina ---
df = pd.DataFrame(results_fine)
print("\n" + "="*60)
print("RESULTADOS PARA ANÁLISE DETALHADA")
print("="*60)
print(df.to_string(index=False))
print("="*60)

--> Testando CUDA Block Size = 128...
--> Testando CUDA Block Size = 256...
--> Testando CUDA Block Size = 512...
--> Testando OpenMP Schedule = static...
--> Testando OpenMP Schedule = dynamic...

RESULTADOS PARA ANÁLISE DETALHADA
         Teste Tempo (ms)                       Obs
CUDA Block 128      24.37            Impacto na GPU
CUDA Block 256      24.62            Impacto na GPU
CUDA Block 512      24.59            Impacto na GPU
 OpenMP Static      48.46 Impacto do Load Balancing
OpenMP Dynamic      91.88 Impacto do Load Balancing


In [17]:
%%writefile kmeans_convergence.c
// Versão modificada apenas para imprimir o SSE a cada passo
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <string.h>

// ... (Inclua as funções de leitura read_csv_1col aqui ou assuma que o compilador linka se for o mesmo .c,
// mas para garantir no Colab, vou colocar o main simplificado que reusa a lógica se você colar isso no lugar do main do naive)
// Para facilitar, vou te dar o bloco COMPLETO desta versão de teste rápido:

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>

// Funções de leitura minificadas
static int count_rows(const char *p){FILE *f=fopen(p,"r");if(!f)return 0;int r=0;char l[8192];while(fgets(l,8192,f)){if(l[0]!='\n')r++;}fclose(f);return r;}
static double *read_csv(const char *p, int *n){int R=count_rows(p);double *A=malloc(R*8);FILE *f=fopen(p,"r");char l[8192];int i=0;while(fgets(l,8192,f)){if(l[0]!='\n')A[i++]=atof(l);}fclose(f);*n=i;return A;}

int main(int argc, char **argv) {
    if(argc<3) return 1;
    int N, K;
    double *X = read_csv(argv[1], &N);
    double *C = read_csv(argv[2], &K);
    int max_iter=50; double eps=0.0001;
    int *assign = malloc(N*4);

    printf("\n--- DADOS PARA O GRÁFICO DE CONVERGÊNCIA ---\n");
    printf("Iteracao,SSE\n");

    double prev_sse = 1e300;
    for(int it=0; it<max_iter; it++){
        double sse = 0.0;
        for(int i=0; i<N; i++){
            double bestd = 1e300; int best = -1;
            for(int c=0; c<K; c++){
                double d = (X[i]-C[c])*(X[i]-C[c]);
                if(d<bestd){bestd=d; best=c;}
            }
            assign[i] = best;
            sse += bestd;
        }

        // AQUI ESTÁ O QUE FALTA:
        printf("%d,%.6f\n", it+1, sse);

        if(fabs(sse-prev_sse)/(prev_sse>0?prev_sse:1) < eps) break;
        prev_sse = sse;

        // Update
        double *sum = calloc(K,8); int *cnt = calloc(K,4);
        for(int i=0; i<N; i++){ sum[assign[i]]+=X[i]; cnt[assign[i]]++; }
        for(int c=0; c<K; c++) C[c] = (cnt[c]>0) ? sum[c]/cnt[c] : X[0];
        free(sum); free(cnt);
    }
    printf("-------------------------------------------\n");
    return 0;
}

Writing kmeans_convergence.c


In [18]:
%%shell
gcc -O2 kmeans_convergence.c -o kmeans_conv -lm
./kmeans_conv dados.csv centroides.csv


--- DADOS PARA O GRÁFICO DE CONVERGÊNCIA ---
Iteracao,SSE
1,112419754.979459
2,43638498.790859
3,43526000.699481
4,43473526.207674
5,43451307.265581
6,43442153.069656
7,43438539.710107
-------------------------------------------
